# Validation

Main pipeline notebook. All logic lives in `pipeline/`; this notebook handles configuration and orchestration only.

## 1. Setup

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
%reload_ext autoreload

In [3]:
import pandas as pd
import os
import sys

from pathlib import Path
from sklearn.metrics import cohen_kappa_score

# Moving up to the project root to ensure imports work correctly regardless of execution context
project_root = Path(os.getcwd()).parent
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))


from pathlib import Path
from openai import OpenAI
from utilities import (
    OPENROUTER_API_KEY,
    OPENROUTER_BASE_URL,
    TASK_STATEMENTS_PATH,
    MAJOR_CATEGORIES,
    WORK_RELATED_EVAL_OUTPUT_PATH,
    TIMEZONES_EVAL_OUTPUT_PATH,
    TASK_MAPPING_EVAL_OUTPUT_PATH,
    LABOR_TRANSFER_EVAL_OUTPUT_PATH,
    JOB_ZONES_PATH,
    ExecutionMode,
    FINAL_EVAL_OUTPUT_PATH,
    EVALUATION_DIR,
)
from pipeline import (
    load_wildchat,
    sample_conversations,
    preprocess_conversations,
    filter_work_conversations,
    find_timezones,
    normalize_timezone,
    map_conversation_to_task,
    filter_task_mappings,
    analyze_labor_transfer,
    expand_labor_transfer_labels,
    analyze_patents,
)
from validation import (
    work_related_metrics,
    agreement_rate,
    print_metrics,
)

In [4]:
# API client setup
client = OpenAI(api_key=OPENROUTER_API_KEY, base_url=OPENROUTER_BASE_URL)

In [5]:
execution_mode = ExecutionMode.DIRECT

## 2. Data Loading

In [6]:
english_conversations = load_wildchat()
total_rows = len(english_conversations)
print(f"Total English conversations: {total_rows}")

Loading dataset from disk:   0%|          | 0/46 [00:00<?, ?it/s]

Total English conversations: 1679371


In [7]:
sample_df = sample_conversations(english_conversations, sample_percentage=0.00006)
print(f"Sample shape: {sample_df.shape}")

Sample shape: (100, 14)


In [8]:
sample_unique_df = preprocess_conversations(sample_df)
print(f"After dedup: {sample_unique_df.shape}")
sample_unique_df.head(2)

After dedup: (100, 5)


,conversation,timestamp,country,state,hashed_ip
0,"[{'role': 'user', 'content': 'can you name any...",2025-03-12 09:33:27,United Kingdom,Royal Kensington and Chelsea,62d519d4f9104d3ad08f25664bf790a39805b5d1e39c21...
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...


In [9]:
sample_conversations_df = sample_unique_df.copy()
sample_conversations_df.shape

(100, 5)

## 3. Work-Related Conversation Filtering

In [10]:
if not WORK_RELATED_EVAL_OUTPUT_PATH.exists():
    answers = filter_work_conversations(
        client=client,
        conversations=sample_conversations_df,
        path=WORK_RELATED_EVAL_OUTPUT_PATH,
        execution_mode=execution_mode,
    )
    answers_df = pd.DataFrame(
        {
            "conversation": sample_conversations_df["conversation"],
            "is_work_related_model": answers,
        }
    )
    answers_df.to_csv(WORK_RELATED_EVAL_OUTPUT_PATH, index=False)
else:
    answers_df = pd.read_csv(WORK_RELATED_EVAL_OUTPUT_PATH)

sample_conversations_df["is_work_related_model"] = answers_df[
    "is_work_related_model"
].values

work_related_df = sample_conversations_df[
    sample_conversations_df["is_work_related_model"] == "Yes"
].copy()

print(f"Work-related conversations: {work_related_df.shape}")
work_related_df.head(2)

Work-related conversations: (31, 6)


,conversation,timestamp,country,state,hashed_ip,is_work_related_model
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...,Yes
5,"[{'role': 'user', 'content': 'System: You are ...",2024-11-04 01:14:03,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes


In [11]:
human_sample = EVALUATION_DIR / "work_related_human.csv"
human_labels_df = pd.read_csv(human_sample)
sample_conversations_df["is_work_related_human"] = human_labels_df[
    "is_work_related_human"
].values

In [12]:
sample_conversations_df["is_work_related_model"].value_counts()

is_work_related_model
Maybe    37
No       32
Yes      31
Name: count, dtype: int64

In [13]:
sample_conversations_df["is_work_related_human"].value_counts()

is_work_related_human
No       50
Yes      32
Maybe    18
Name: count, dtype: int64

In [14]:
no_array = ["No"] * len(sample_conversations_df)
no_array_df = pd.DataFrame({"answer": no_array})

In [15]:
metrics = work_related_metrics(
    y_true=sample_conversations_df["is_work_related_human"],
    y_pred=sample_conversations_df["is_work_related_model"],
)
print_metrics(metrics)

accuracy: 0.7700
fpr: 0.0294
tpr: 0.9062
precision: 0.9355
recall: 0.9062
cohen_kappa: 0.6589


In [17]:
no_array_df

,answer
0,No
1,No
2,No
3,No
4,No
...,...
95,No
96,No
97,No
98,No


In [18]:
sample_conversations_df["is_work_related_human"].value_counts()

is_work_related_human
No       50
Yes      32
Maybe    18
Name: count, dtype: int64

In [16]:
metrics = work_related_metrics(
    y_true=sample_conversations_df["is_work_related_human"],
    y_pred=no_array_df["answer"],
)
print_metrics(metrics)

accuracy: 0.5000
fpr: 0.0000
tpr: 0.0000
precision: 0
recall: 0.0000
cohen_kappa: 0.0000


In [14]:
disagreements_df = sample_conversations_df[
    sample_conversations_df["is_work_related_human"]
    != sample_conversations_df["is_work_related_model"]
].copy()
disagreements_df.to_csv("disagreements.csv", index=False)

In [15]:
sample_conversations_df = sample_conversations_df[
    sample_conversations_df["is_work_related_model"] == "Yes"
].copy()
sample_conversations_df.shape

(31, 7)

## 4. Timezone conversion

In [16]:
work_related_df.head(5)

,conversation,timestamp,country,state,hashed_ip,is_work_related_model
1,"[{'role': 'user', 'content': 'sir followings a...",2024-10-01 15:01:08,Pakistan,Khyber Pakhtunkhwa,65ee10fbdef827a0b2eeb88fd635b256ce0b54ddf02c73...,Yes
5,"[{'role': 'user', 'content': 'System: You are ...",2024-11-04 01:14:03,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes
8,"[{'role': 'user', 'content': 'You are a helpfu...",2024-11-05 23:54:06,Denmark,Capital Region,1b8b85252359a2c246c3086a72367c7448601985693934...,Yes
10,"[{'role': 'user', 'content': 'You are a helpfu...",2024-11-09 09:44:50,Belgium,Brussels Capital,5a527b1cb1af88eb990a824042cb105bb336ee03606f4f...,Yes
11,"[{'role': 'user', 'content': 'hello '}, {'role...",2023-05-15 12:30:47,Rwanda,Eastern Province,9b79c742926f06b75a7913e23f49334e593ed5ed04625a...,Yes


In [17]:
if not TIMEZONES_EVAL_OUTPUT_PATH.exists():
    work_related_df = find_timezones(df=work_related_df)
    work_related_df.to_csv(TIMEZONES_EVAL_OUTPUT_PATH, index=False)
else:
    work_related_df = pd.read_csv(TIMEZONES_EVAL_OUTPUT_PATH)

if "timezone_y" in work_related_df.columns:
    work_related_df.drop(columns=["timezone_y"], inplace=True)
    work_related_df = work_related_df.rename(columns={"timezone_x": "timezone"})
work_related_df = work_related_df[work_related_df["timezone"].notnull()]
print(f"After timezone filter: {work_related_df.shape}")

Geocoding query: 'Khyber Pakhtunkhwa, Pakistan' -> Location: خیبر پختونخوا, پاکستان
Found location for query 'Khyber Pakhtunkhwa, Pakistan': خیبر پختونخوا, پاکستان (lat: 33.712802, lng: 71.2678805)
Geocoding query: 'nan' -> Location: ننگرهار ولايت, افغانستان
Found location for query 'nan': ننگرهار ولايت, افغانستان (lat: 34.220389, lng: 70.3800314)
Geocoding query: 'Capital Region, Denmark' -> Location: Sendiráð Danmerkur, 29, Hverfisgata, Austurbær, Miðborg, Reykjavíkurborg, Höfuðborgarsvæðið, 101, Ísland
Found location for query 'Capital Region, Denmark': Sendiráð Danmerkur, 29, Hverfisgata, Austurbær, Miðborg, Reykjavíkurborg, Höfuðborgarsvæðið, 101, Ísland (lat: 64.146742, lng: -21.92952)
Geocoding query: 'Brussels Capital, Belgium' -> Location: Région de Bruxelles-Capitale - Brussels Hoofdstedelijk Gewest, België / Belgique / Belgien
Found location for query 'Brussels Capital, Belgium': Région de Bruxelles-Capitale - Brussels Hoofdstedelijk Gewest, België / Belgique / Belgien (lat:

In [18]:
work_related_df = normalize_timezone(df=work_related_df)
work_related_df[["timestamp", "timezone", "timestamp_local"]].head(3)

,timestamp,timezone,timestamp_local
0,2024-10-01 15:01:08+00:00,Asia/Karachi,2024-10-01 20:01:08+05:00
1,2024-11-04 01:14:03+00:00,Asia/Kabul,2024-11-04 05:44:03+04:30
2,2024-11-05 23:54:06+00:00,Atlantic/Reykjavik,2024-11-05 23:54:06+00:00


In [19]:
work_related_df["conversation"].head(5)

0    [{'role': 'user', 'content': 'sir followings a...
1    [{'role': 'user', 'content': 'System: You are ...
2    [{'role': 'user', 'content': 'You are a helpfu...
3    [{'role': 'user', 'content': 'You are a helpfu...
4    [{'role': 'user', 'content': 'hello '}, {'role...
Name: conversation, dtype: object

## 5. Task Mapping

In [20]:
tasks_df = pd.read_csv(TASK_STATEMENTS_PATH)
tasks_df.drop(
    columns=["Incumbents Responding", "Date", "Domain Source", "Task Type"],
    inplace=True,
)
tasks_df["major_category"] = tasks_df["O*NET-SOC Code"].apply(
    lambda x: MAJOR_CATEGORIES[x[0:2]]
)
tasks_df.head()

,O*NET-SOC Code,Title,Task ID,Task,major_category
0,11-1011.00,Chief Executives,8823,Direct or coordinate an organization's financi...,Management Occupations
1,11-1011.00,Chief Executives,8831,Appoint department heads or managers and assig...,Management Occupations
2,11-1011.00,Chief Executives,8825,Analyze operations to evaluate performance of ...,Management Occupations
3,11-1011.00,Chief Executives,8826,"Direct, plan, or implement policies, objective...",Management Occupations
4,11-1011.00,Chief Executives,8827,"Prepare budgets for approval, including those ...",Management Occupations


In [21]:
if not TASK_MAPPING_EVAL_OUTPUT_PATH.exists():
    task_mapped_df = map_conversation_to_task(
        client=client,
        conversations=work_related_df,
        tasks=tasks_df,
        path=TASK_MAPPING_EVAL_OUTPUT_PATH,
        execution_mode=execution_mode,
    )

    task_mapped_df = filter_task_mappings(df=task_mapped_df, column_name="tasks")

    task_mapped_df["job_title"] = task_mapped_df["tasks"].apply(
        lambda x: x.split(":")[0] if pd.notnull(x) else None
    )
    task_mapped_df["selected_task"] = task_mapped_df["tasks"].apply(
        lambda x: x.split(":")[1] if pd.notnull(x) else None
    )

    print(f"After consensus filter: {task_mapped_df.shape}")

    task_mapped_df["conversation_str"] = task_mapped_df["conversation"].astype(str)
    work_related_df["conversation_str"] = work_related_df["conversation"].astype(str)

    task_mapped_df = task_mapped_df.merge(
        work_related_df.drop(columns=["conversation"]),
        on="conversation_str",
        how="inner",
    )

    task_mapped_df = task_mapped_df.drop(columns=["conversation_str"])

    print(f"Final task mapped DataFrame: {task_mapped_df.shape}")
    task_mapped_df.head(2)
else:
    task_mapped_df = pd.read_csv(TASK_MAPPING_EVAL_OUTPUT_PATH)

print(f"Task mapped conversations: {task_mapped_df.shape}")
task_mapped_df.head(2)

Conversations to process: [[{'role': 'user', 'content': 'sir followings are the enssential items/equpments listed for office . correct the following'}, {'role': 'assistant', 'content': 'Of course! Please provide the list of essential items/equipment you would like me to review and correct.'}, {'role': 'user', 'content': 'yes'}, {'role': 'assistant', 'content': "Please go ahead and provide the list, and I'll help you with any corrections needed."}, {'role': 'user', 'content': 'correct the sentance '}, {'role': 'assistant', 'content': 'Sure, here\'s the corrected sentence:\n\n"Sir, the following are the essential items/equipment listed for the office. Please correct the following."\n\nWould you like further assistance with anything else?'}, {'role': 'user', 'content': "add kitchen , It Equpment's and stationary "}, {'role': 'assistant', 'content': 'Certainly, here is a revised version of your sentence with those additions:\n\n"Sir, the following are the essential items/equipment listed f

Profession Mapping: 100%|██████████| 30/30 [01:57<00:00,  3.93s/it]


Running 30 direct Task mapping calls via OpenRouter...


Task Mapping: 100%|██████████| 30/30 [02:32<00:00,  5.10s/it]

After consensus filter: (13, 5)
Final task mapped DataFrame: (13, 12)
Task mapped conversations: (13, 12)


,conversation,professions,tasks,job_title,selected_task,timestamp,country,state,hashed_ip,is_work_related_model,timezone,timestamp_local
0,"[{'role': 'user', 'content': 'System: You are ...","[Interpreters and Translators, Technical Write...",Interpreters and Translators:Read written mate...,Interpreters and Translators,"Read written materials, such as legal document...",2024-11-04 01:14:03+00:00,NaN,NaN,738c8bec3e3b3fe22bedc4885fb26c230754e22747e031...,Yes,Asia/Kabul,2024-11-04 05:44:03+04:30
1,"[{'role': 'user', 'content': 'System: IMPORTAN...","[Human Resources Specialists, Compensation, Be...",Human Resources Specialists:Review employment ...,Human Resources Specialists,Review employment applications and job orders ...,2024-11-06 01:44:28+00:00,Germany,NaN,968b0a296a779efbb3a2a167a12a3769fb3f6c8ad46329...,Yes,Europe/Berlin,2024-11-06 02:44:28+01:00


In [22]:
task_mapped_df.to_csv(TASK_MAPPING_EVAL_OUTPUT_PATH, index=False)

In [25]:
agreement_human = EVALUATION_DIR / "task_mapping_human_eval.csv"
task_agreement_df = pd.read_csv(agreement_human)

In [26]:
task_mapped_df["selected_task_human_eval"] = task_agreement_df[
    "selected_task_human_eval"
]

In [27]:
agreement = agreement_rate(
    df=task_mapped_df,
    column="selected_task_human_eval",
)
print(f"Human evaluators agreement with the task assignment: {agreement}")

Human evaluators agreement with the task assignment: 0.9230769230769231


## 6. Labor Transfer Analysis

In [28]:
labor_transfer_df = pd.read_csv(TASK_MAPPING_EVAL_OUTPUT_PATH)
labor_transfer_df.shape

(13, 12)

In [29]:
if not LABOR_TRANSFER_EVAL_OUTPUT_PATH.exists():
    labor_transfer_labels = analyze_labor_transfer(
        client=client, df=task_mapped_df, execution_mode=execution_mode
    )
    pd.DataFrame({"label": labor_transfer_labels}).to_csv(
        LABOR_TRANSFER_EVAL_OUTPUT_PATH, index=False
    )
    print(f"Labor transfer labels: {len(labor_transfer_labels)}")
    labor_transfer_df["labor_transfer"] = labor_transfer_labels
    labor_transfer_df = expand_labor_transfer_labels(
        df=labor_transfer_df, label_column="labor_transfer"
    )
    print(f"Final DataFrame: {labor_transfer_df.shape}")
    labor_transfer_df.to_csv(LABOR_TRANSFER_EVAL_OUTPUT_PATH, index=False)
    labor_transfer_df.head(2)
else:
    labor_transfer_df = pd.read_csv(LABOR_TRANSFER_EVAL_OUTPUT_PATH)


print(f"Labor transfer labels assigned: {labor_transfer_df.shape}")

Running 13 direct Labor Transfer calls via OpenRouter...


Labor Transfer: 100%|██████████| 13/13 [00:35<00:00,  2.72s/it]

Labor transfer labels: 13
Final DataFrame: (13, 20)
Labor transfer labels assigned: (13, 20)


In [30]:
labor_transfer_human = EVALUATION_DIR / "labor_transfer_human_eval.csv"
labor_transfer_human_df = pd.read_csv(labor_transfer_human)

In [31]:
labor_transfer_df["labor_transfer_human_eval"] = labor_transfer_human_df[
    "labor_transfer_human_eval"
]

In [50]:
cohen = cohen_kappa_score(
    labor_transfer_df["label"],
    labor_transfer_df["labor_transfer_human_eval"],
)
print(f"Cohen's kappa score for labor transfer labels: {cohen}")

Cohen's kappa score for labor transfer labels: 0.8194444444444444


In [54]:
import numpy as np

unique_classes = labor_transfer_df["label"].unique()

np.random.seed(42)
predizioni_random = np.random.choice(unique_classes, size=len(labor_transfer_df))

random_baseline_df = pd.DataFrame({"answer": predizioni_random})

cohen = cohen_kappa_score(labor_transfer_df["label"], random_baseline_df["answer"])
print(f"Cohen's kappa score for Random Baseline: {cohen:.4f}")

Cohen's kappa score for Random Baseline: 0.0647
